# 01 · SWORD extraction

Traces each river through SWORD v17b from its upstream to its downstream
reach ID and writes the traced reaches and nodes to
`{data_root}/{river}/sword/`.

**Before running:** find the two reach IDs for each river in QGIS (load
`{continent}_sword_reaches_v17b.gpkg`) and add a row to the rivers table:
`river_name, sword_cont_abb, us_sword_reach_id, ds_sword_reach_id`.

The logic lives in `rivmapper_v2/sword.py`; this notebook only runs it and
shows QC plots. Same thing from a terminal:

```
rivmapper run projects/oktedi/rivmapper.yaml --steps sword
```

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt

from rivmapper_v2 import Project, run_pipeline, setup_logging

setup_logging()

# ── Settings ─────────────────────────────────────────────────────────
# Everything else lives in rivmapper.yaml and the rivers table it points to.
CONFIG_PATH = Path("../projects/oktedi/rivmapper.yaml")  # relative to notebooks/
RIVERS = None  # None = all rivers, or e.g. ["Beni", "Bermejo"]
OVERWRITE = False

In [ ]:
project = Project.load(CONFIG_PATH)
if OVERWRITE:
    project.rivers = [r.model_copy(update={"overwrite": True}) for r in project.rivers]
print(project)
print("Rivers:", ", ".join(project.river_names))

## Run

In [ ]:
results = run_pipeline(project, steps=["sword"], rivers=RIVERS)
results

## QC: traced path per river

Colour runs upstream (dark) → downstream (light). Check the path follows the main channel, especially where SWORD splits (the tracer always takes the first downstream branch).

In [ ]:
for river in project.select(RIVERS):
    p = river.paths
    if not p.sword_reaches.exists():
        continue
    reaches = gpd.read_file(p.sword_reaches)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    reaches.plot(ax=ax, column="reach_order", cmap="viridis", linewidth=1.5)
    ax.set_title(f"{river.name}: {len(reaches)} SWORD reaches")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.show()